## Segunda parte do tratamento da base de dados do Bolsa Família

#### 1 - Abrir o arquivo bolsafamilia tratado em parquet para continuar o tratamento e treinamento:

In [ ]:
import polars as pl
import plotly.express as px

bolsa2 = "bolsapl_tratado.parquet"

df_bolsa2 = pl.scan_parquet(bolsa2)

df_bolsa2

### 2 - Verifica a quantidade de beneficiários (usuários) únicos existentes na base:

In [10]:
df_bolsa2.select(
    pl.col("nis").n_unique().alias("beneficiarios_unicos")
).collect()

beneficiarios_unicos
u32
20486709


### 3 - Verificar o esquema de colunas da base, de quais tipos são:

In [11]:
df_bolsa2.collect_schema()

Schema([('mes_comp', Int64),
        ('mes_ref', Int64),
        ('uf', String),
        ('cod_municipio', Int64),
        ('municipio', String),
        ('nis', String),
        ('favorecido', String),
        ('parcela', Float64)])

### 4 - Realizar as estatísticas descritivas da coluna 'parcela', usando o modo lazy:

In [ ]:
df_bolsa2.select([
    pl.col("parcela").mean().alias("media"),
    pl.col("parcela").median().alias("mediana"),
    pl.col("parcela").min().alias("minimo"),
    pl.col("parcela").max().alias("maximo"),
    pl.col("parcela").std().alias("desvio_padrao"),
    pl.col("parcela").sum().alias("total_pago")
]).collect()

media,mediana,minimo,maximo,desvio_padrao,total_pago
f64,f64,f64,f64,f64,f64
670.091255,650.0,25.0,3938.0,189.928508,2.7347e10


### 5 - Criando faixas de valor para as parcelas e verificar quantos beneficiários se encaixam em cada uma delas:

In [13]:
df_faixas = (
    df_bolsa2
    .with_columns(
        pl.when(pl.col("parcela") < 200).then(pl.lit("0-199"))
        .when(pl.col("parcela") < 400).then(pl.lit("200-399"))
        .when(pl.col("parcela") < 600).then(pl.lit("400-599"))
        .when(pl.col("parcela") < 800).then(pl.lit("600-799"))
        .when(pl.col("parcela") < 1000).then(pl.lit("800-999"))
        .otherwise(pl.lit("1000+"))
        .alias("faixa_valor")
    )
    .group_by("faixa_valor")
    .agg([
        pl.col("parcela").count().alias("quantidade"),
        pl.col("parcela").mean().alias("media_faixa")
    ])
)

df_faixas = df_faixas.collect()

ordem_correta = ["0-199", "200-399", "400-599", "600-799", "800-999", "1000+"]
ordem_dict = {v: i for i, v in enumerate(ordem_correta)}

df_faixas = (
    df_faixas
    .with_columns(
        pl.col("faixa_valor")
        .replace(ordem_dict)
        .alias("ordem")
    )
    .sort("ordem")
    .drop("ordem")
)

df_faixas

faixa_valor,quantidade,media_faixa
str,u32,f64
"""0-199""",845,46.775148
"""200-399""",4156101,330.172578
"""400-599""",1564080,443.385141
"""600-799""",26235366,653.384646
"""800-999""",7015353,850.473527
"""1000+""",1839248,1181.55477


### 6 - Verificando a estrutura do LazyFrame:

In [ ]:
df_bolsa2.shape
df_bolsa2.describe()

statistic,mes_comp,mes_ref,uf,cod_municipio,municipio,nis,favorecido,parcela
str,f64,f64,str,f64,str,str,str,f64
"""count""",4.0810993e7,4.0810993e7,"""40810993""",4.0810993e7,"""40810993""","""40810268""","""40810993""",4.0810993e7
"""null_count""",0.0,0.0,"""0""",0.0,"""0""","""725""","""0""",0.0
"""mean""",202501.499349,202501.124097,null,3837.013512,null,null,null,670.091255
"""std""",0.5,5.844027,null,2801.632256,null,null,null,189.928508
"""min""",202501.0,202308.0,"""AC""",1.0,"""abadiadegoias""","""10000612860""","""*** BENEFICIÁRIO MENOR DE 16 A…",25.0
"""25%""",202501.0,202501.0,null,1389.0,null,null,null,600.0
"""50%""",202501.0,202501.0,null,3351.0,null,null,null,650.0
"""75%""",202502.0,202502.0,null,6001.0,null,null,null,750.0
"""max""",202502.0,202502.0,"""TO""",9997.0,"""zortea""","""27292246805""","""ÎNGRID GONCALVES SANTOS""",3938.0


### 7 - Criar gráfico de barras apresentando a quantidade de beneficiários por faixa etária, baseado no código anterior:

In [25]:
fig = px.bar(
    df_faixas.to_pandas(),
    x="quantidade",
    y="faixa_valor",
    orientation="h",
    text="quantidade",
    title="Distribuição de Beneficiários por Faixa de Valor do Bolsa Família",
    color="faixa_valor",
    color_discrete_sequence=px.colors.sequential.Tealgrn
)

fig.update_traces(
    texttemplate='%{text:,.0f}', 
    textposition='outside')
fig.update_layout(
    xaxis_title="Número de Beneficiários",
    yaxis_title="Faixa de Valor",
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=True),
    yaxis=dict(categoryorder='array', categoryarray=df_faixas["faixa_valor"].to_list())
)

fig.show()

fig.write_html("grafico_beneficiarios_por_faixa_etaria.html")

### 7 - Fazendo um comparativo do total pago entre os meses de janeiro e fevereiro:

In [27]:
totais_por_mes = (
    df_bolsa2
    .group_by("mes_comp")
    .agg(pl.col("parcela").sum().alias("total_pago"))
    .sort("mes_comp")
)

totais_por_mes

mes_comp,total_pago
i64,f64
202501,1.3693e10
202502,1.3654e10


### 8 - Comparar os beneficiários que permaneceram, entraram ou saíram de um mês pro outro:

In [32]:
meses = df_bolsa2.select("mes_comp").unique().to_series().sort().to_list()
mes1, mes2 = meses

df_mes1 = df_bolsa2.filter(pl.col("mes_comp") == mes1).select(["nis", "favorecido"]).unique()
df_mes2 = df_bolsa2.filter(pl.col("mes_comp") == mes2).select(["nis", "favorecido"]).unique()

df_continuaram = df_mes1.join(df_mes2, on=["nis", "favorecido"], how="inner")

df_sairam = df_mes1.join(df_mes2, on=["nis", "favorecido"], how="anti")

df_novos = df_mes2.join(df_mes1, on=["nis", "favorecido"], how="anti")

resumo = pl.DataFrame({
    "categoria": ["Continuaram", "Saíram", "Entraram"],
    "quantidade": [
        df_continuaram.height,
        df_sairam.height,
        df_novos.height
    ]
})

resumo

categoria,quantidade
str,i64
"""Continuaram""",20106565
"""Saíram""",208012
"""Entraram""",195707


### 9 - Verificar os 10 beneficiários que obtiveram o maior aumento no valor do benefício de um mês para o outro e em sequência os que obtiveram os maiores cortes:

In [42]:
df_mes1 = df_bolsa2.filter(pl.col("mes_comp") == mes1)
df_mes2 = df_bolsa2.filter(pl.col("mes_comp") == mes2)

media_mes1 = (
    df_mes1.group_by("favorecido")
    .agg(pl.col("parcela").mean().alias("media_mes1"))
)

media_mes2 = (
    df_mes2.group_by("favorecido")
    .agg(pl.col("parcela").mean().alias("media_mes2"))
)

comparacao = (
    media_mes1.join(media_mes2, on="favorecido", how="inner")
    .with_columns((pl.col("media_mes2") - pl.col("media_mes1")).alias("diferenca"))
    .sort("diferenca", descending=True)
)

comparacao.head(10)


favorecido,media_mes1,media_mes2,diferenca
str,f64,f64,f64
"""KATIA LOBAO DA CONCEICAO""",300.0,2078.0,1778.0
"""JESSICA CASSIANA RODRIGUES""",425.0,1962.0,1537.0
"""DANIELI APARECIDA GASS""",968.0,2470.0,1502.0
"""JULIANA JUCELI MACHADO DE ALME…",601.0,1928.0,1327.0
"""SULEIDE FARIA CRISANTO""",850.0,2136.0,1286.0
"""ANTONIA MARIA FEITOSA DOS SANT…",300.0,1578.0,1278.0
"""SEBASTIANA APARECIDA LUCIO OLI…",743.0,2020.0,1277.0
"""MARCILIANE APARECIDA AMBROSIO""",889.0,2120.0,1231.0
"""CRISTIANE SANTOS DE CASTRO""",325.0,1542.5,1217.5


In [43]:
df_mes1 = df_bolsa2.filter(pl.col("mes_comp") == mes1)
df_mes2 = df_bolsa2.filter(pl.col("mes_comp") == mes2)

media_mes1 = (
    df_mes1.group_by("favorecido")
    .agg(pl.col("parcela").mean().alias("media_mes1"))
)

media_mes2 = (
    df_mes2.group_by("favorecido")
    .agg(pl.col("parcela").mean().alias("media_mes2"))
)

comparacao = (
    media_mes1.join(media_mes2, on="favorecido", how="inner")
    .with_columns((pl.col("media_mes2") - pl.col("media_mes1")).alias("diferenca"))
    .sort("diferenca", descending=True)
)

comparacao.sort("diferenca").head(10)

favorecido,media_mes1,media_mes2,diferenca
str,f64,f64,f64
"""FRANCISCO ERICO DE SOUSA ARAUJ…",2998.0,710.0,-2288.0
"""EDSON DE JESUS SOARES""",3146.0,1152.0,-1994.0
"""MILIANE SILVA DE BRITO""",2478.0,750.0,-1728.0
"""JOSIANI SEGUNDO CACIANO SANTAN…",2028.0,450.0,-1578.0
"""MARILENE DIAS NASCIMENTO""",2404.0,834.666667,-1569.333333
"""MARIA DAS GRACAS ALMEIDA DE BR…",2036.0,600.0,-1436.0
"""GRACIANA DE BRITO FERREIRA""",2186.0,750.0,-1436.0
"""RONALDE ALMEIDA MACIEL""",2036.0,600.0,-1436.0
"""GENEIDE SILVA SANTOS""",2304.0,940.8,-1363.2


### 10 - Gráfico que apresenta estas informações acima:

In [45]:
top_aumentos = comparacao.sort("diferenca", descending=True).head(10)
top_cortes = comparacao.sort("diferenca", descending=False).head(10)

comparacao_grafico = pl.concat([top_aumentos, top_cortes])

comparacao_pd = comparacao_grafico.to_pandas()

fig = px.bar(
    comparacao_pd,
    x="favorecido",
    y="diferenca",
    color="diferenca",
    color_continuous_scale=["red", "gray", "green"],
    title="Diferença no valor médio por favorecido entre os meses",
)

fig.update_layout(
    xaxis_title="Favorecido",
    yaxis_title="Diferença (R$)",
    xaxis_tickangle=-45,
    showlegend=False,
)

fig.show()